# 🏈 Probabilidades semanales in-season · Temporada 2026

Este notebook actualiza los ratings ELO **partido a partido usando solo
resultados reales ya ocurridos**, y calcula la probabilidad de victoria de
cada juego con el ELO vigente a esa fecha de la temporada. A diferencia del
Notebook 04 (Monte Carlo sobre toda la temporada asumiendo que nada se ha
jugado), aquí la lógica es:

```
Semana 1 -> ELOs de cierre de 2025 + regresión de temporada  (ya lo teníamos)
Semana 2 -> ELOs actualizados con los resultados reales de la semana 1
Semana 3 -> ELOs actualizados con los resultados reales de las semanas 1-2
...
```

`nflreadpy` va agregando resultados reales conforme se juegan los partidos,
así que basta con **volver a correr este notebook cada semana** para obtener
las probabilidades actualizadas de la siguiente semana — sin volver a correr
Monte Carlo completo.

La lógica de ELO se reutiliza de `src/elo.py` (mismas funciones y parámetros
calibrados en el Notebook 02). El motor de este notebook vive en
`src/weekly_probabilities.py`.

In [1]:
# ── §1 · Setup ────────────────────────────────────────────────────────────────
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import nflreadpy as nfl
import plotly.graph_objects as go
import plotly.express as px

from pathlib import Path

from src.weekly_probabilities import (
    build_weekly_snapshots,
    current_week_probabilities,
    team_schedule_view,
)
from src.simulation import TEAM_CONF, TEAM_DIV

# Parámetros calibrados en Notebook 02 (mismos que Notebook 04)
HOME_ADVANTAGE    = 42.2
K_REGULAR         = 40
K_PLAYOFFS        = 48
REGRESSION_FACTOR = 0.40
GLOBAL_MEAN       = 1503.1

SEASON = 2026

# Paleta consistente con src/visualization.py
AFC_COLOR  = '#013369'
NFC_COLOR  = '#D50A0A'
GOLD_COLOR = '#f5c518'

In [2]:
# ── §2 · Cargar datos ─────────────────────────────────────────────────────────
# Calendario de la temporada — nflreadpy trae los marcadores reales conforme
# se van jugando los partidos. Antes de que empiece la temporada, home_score/
# away_score vienen vacíos para todos los juegos.
schedule = nfl.load_schedules(seasons=[SEASON]).to_pandas()

# ELOs finales de la temporada anterior (Notebook 02)
final_elos_df = pd.read_csv('../data/processed/elo_final_2025.csv')
final_elos    = dict(zip(final_elos_df['team'], final_elos_df['elo_final']))

n_played  = schedule[['home_score', 'away_score']].notna().all(axis=1).sum()
n_total   = len(schedule)
print(f"Partidos {SEASON}      : {n_total}")
print(f"Ya jugados          : {n_played}")
print(f"Pendientes           : {n_total - n_played}")
print(f"Equipos con ELO base : {len(final_elos)}")

Partidos 2026      : 272
Ya jugados          : 32
Pendientes           : 240
Equipos con ELO base : 32


In [3]:
# ── §3 · Construir snapshots semanales ────────────────────────────────────────
# games_df        : un renglón por partido, con ELO pre/post y probabilidad
#                    de victoria del local calculada ANTES de que se jugara.
# elo_evolution_df: ELO de cada equipo al inicio de cada semana (para graficar
#                    trayectorias a lo largo de la temporada).
games_df, elo_evolution_df = build_weekly_snapshots(
    schedule_df=schedule,
    final_elos_prev_season=final_elos,
    home_advantage=HOME_ADVANTAGE,
    k_regular=K_REGULAR,
    k_playoffs=K_PLAYOFFS,
    regression_factor=REGRESSION_FACTOR,
    global_mean=GLOBAL_MEAN,
)

print(f"games_df: {games_df.shape}  ·  elo_evolution_df: {elo_evolution_df.shape}")
games_df.head(20)

games_df: (272, 15)  ·  elo_evolution_df: (576, 3)


,season,week,game_type,home_team,away_team,elo_home_pre,elo_away_pre,p_home_pre,p_away_pre,played,home_score,away_score,result,elo_home_post,elo_away_post
0,2026,1,REG,SEA,NE,1643.35,1592.82,0.6304,0.3696,True,13.0,10.0,1.0,1658.13,1578.03
1,2026,1,REG,LA,SF,1577.37,1559.23,0.5860,0.4140,True,7.0,27.0,0.0,1553.93,1582.67
2,2026,1,REG,CAR,CHI,1457.76,1531.36,0.4549,0.5451,True,37.0,59.0,0.0,1439.56,1549.55
3,2026,1,REG,CIN,TB,1467.06,1476.89,0.5465,0.4535,True,33.0,27.0,1.0,1485.21,1458.75
4,2026,1,REG,DET,NO,1543.71,1451.63,0.6842,0.3158,True,31.0,30.0,1.0,1556.34,1439.00
5,2026,1,REG,HOU,BUF,1581.66,1579.00,0.5642,0.4358,True,31.0,36.0,0.0,1559.09,1601.57
6,2026,1,REG,IND,BAL,1472.33,1518.16,0.4948,0.5052,True,23.0,41.0,0.0,1452.54,1537.95
7,2026,1,REG,JAX,CLE,1545.27,1437.71,0.7031,0.2969,True,34.0,10.0,1.0,1557.14,1425.83
8,2026,1,REG,PIT,ATL,1521.57,1493.61,0.5996,0.4004,True,20.0,13.0,1.0,1537.59,1477.60
9,2026,1,REG,TEN,NYJ,1391.90,1401.98,0.5461,0.4539,True,10.0,23.0,0.0,1370.06,1423.83


In [4]:
# ── §4 · Guardar outputs ───────────────────────────────────────────────────────
out_dir = Path('../data/processed')
weekly_dir = out_dir / 'weekly'
weekly_dir.mkdir(parents=True, exist_ok=True)

games_df.to_csv(out_dir / f'weekly_predictions_{SEASON}.csv', index=False)
elo_evolution_df.to_csv(out_dir / f'elo_evolution_{SEASON}.csv', index=False)

# Un CSV por semana — útil para trackear cómo cambiaron las probabilidades
for week, week_df in games_df.groupby('week'):
    week_df.to_csv(weekly_dir / f'week_{int(week):02d}_probabilities.csv', index=False)

print(f"Guardado: weekly_predictions_{SEASON}.csv, elo_evolution_{SEASON}.csv")
print(f"Guardado: {games_df['week'].nunique()} archivos en data/processed/weekly/")

Guardado: weekly_predictions_2026.csv, elo_evolution_2026.csv
Guardado: 18 archivos en data/processed/weekly/


In [6]:
games_df.query("week == 3").sort_values('p_away_pre', ascending=False)

,season,week,game_type,home_team,away_team,elo_home_pre,elo_away_pre,p_home_pre,p_away_pre,played,home_score,away_score,result,elo_home_post,elo_away_post
41,2026,3,REG,WAS,SEA,1436.48,1668.27,0.2514,0.7486,False,NaN,NaN,NaN,1436.48,1668.27
43,2026,3,REG,TB,MIN,1434.49,1582.77,0.3519,0.6481,False,NaN,NaN,NaN,1434.49,1582.77
36,2026,3,REG,IND,HOU,1437.87,1532.64,0.4249,0.5751,False,NaN,NaN,NaN,1437.87,1532.64
38,2026,3,REG,MIA,KC,1441.51,1519.98,0.4480,0.5520,False,NaN,NaN,NaN,1441.51,1519.98
37,2026,3,REG,JAX,NE,1540.32,1593.36,0.4844,0.5156,False,NaN,NaN,NaN,1540.32,1593.36
47,2026,3,REG,CHI,PHI,1527.79,1580.67,0.4846,0.5154,False,NaN,NaN,NaN,1527.79,1580.67
44,2026,3,REG,DAL,BAL,1471.60,1510.25,0.5051,0.4949,False,NaN,NaN,NaN,1471.60,1510.25
34,2026,3,REG,CLE,CAR,1450.09,1464.10,0.5405,0.4595,False,NaN,NaN,NaN,1450.09,1464.10
40,2026,3,REG,PIT,CIN,1522.26,1511.65,0.5754,0.4246,False,NaN,NaN,NaN,1522.26,1511.65
46,2026,3,REG,DEN,LA,1587.51,1565.51,0.5914,0.4086,False,NaN,NaN,NaN,1587.51,1565.51


## §5 · Probabilidades de la próxima semana sin jugar

`current_week_probabilities()` detecta automáticamente la primera semana
pendiente (la próxima por jugarse) y regresa esos partidos con la
probabilidad de victoria calculada usando el ELO más reciente disponible.

In [7]:
next_week_df = current_week_probabilities(games_df)
next_week_n  = int(next_week_df['week'].iloc[0])
print(f"Próxima semana con probabilidades: Semana {next_week_n}")

plot_df = next_week_df.copy()
plot_df['matchup'] = plot_df['away_team'] + ' @ ' + plot_df['home_team']
plot_df = plot_df.sort_values('p_home_pre')
plot_df['home_color'] = plot_df['home_team'].map(TEAM_CONF).map({'AFC': AFC_COLOR, 'NFC': NFC_COLOR})

fig = go.Figure()
fig.add_trace(go.Bar(
    x=plot_df['p_home_pre'],
    y=plot_df['matchup'],
    orientation='h',
    marker_color=plot_df['home_color'],
    text=[f"{p:.0%}" for p in plot_df['p_home_pre']],
    textposition='outside',
    hovertemplate='%{y}<br>Prob. victoria local: %{x:.1%}<extra></extra>',
))
fig.add_vline(x=0.5, line_dash='dash', line_color='gray')
fig.update_layout(
    title=f'Probabilidad de victoria del local · Semana {next_week_n} · {SEASON}',
    xaxis_title='Prob. victoria del equipo local',
    xaxis_tickformat='.0%',
    xaxis_range=[0, 1],
    template='plotly_white',
    height=max(400, 28 * len(plot_df)),
    margin=dict(l=120),
)
fig.show()

Próxima semana con probabilidades: Semana 3


## §6 · Heatmap — probabilidad de victoria por equipo y semana

Vista de toda la temporada: cada celda es la probabilidad de que ese equipo
gane su partido de esa semana (sin importar si jugó de local o visitante),
calculada con el ELO vigente al momento de jugarse (o el más reciente
disponible, para semanas futuras). El hover muestra rival, local/visitante
y si ya se jugó.

In [8]:
team_views = []
for t in sorted(final_elos.keys()):
    v = team_schedule_view(games_df, t)
    v['team'] = t
    team_views.append(v)

long_df = pd.concat(team_views, ignore_index=True)

prob_pivot   = long_df.pivot(index='team', columns='week', values='team_win_prob')
opp_pivot    = long_df.pivot(index='team', columns='week', values='opponent')
home_pivot   = long_df.pivot(index='team', columns='week', values='is_home')
played_pivot = long_df.pivot(index='team', columns='week', values='played')

# Ordenar equipos por conferencia/división para que el heatmap se lea mejor
team_order = sorted(prob_pivot.index, key=lambda t: (TEAM_CONF[t], TEAM_DIV[t], t))
prob_pivot, opp_pivot, home_pivot, played_pivot = (
    df.loc[team_order] for df in (prob_pivot, opp_pivot, home_pivot, played_pivot)
)

customdata = np.dstack([
    opp_pivot.values,
    np.where(home_pivot.values == True, 'Local', 'Visitante'),
    np.where(played_pivot.values == True, 'Jugado', 'Pendiente'),
])

fig = go.Figure(data=go.Heatmap(
    z=prob_pivot.values,
    x=[f'S{w}' for w in prob_pivot.columns],
    y=prob_pivot.index,
    customdata=customdata,
    colorscale='RdBu',
    zmid=0.5,
    zmin=0, zmax=1,
    colorbar=dict(title='Prob.<br>victoria', tickformat='.0%'),
    hovertemplate=(
        '<b>%{y}</b> · %{x}<br>'
        'Rival: %{customdata[0]} (%{customdata[1]})<br>'
        'Prob. victoria: %{z:.1%}<br>'
        '%{customdata[2]}<extra></extra>'
    ),
))
fig.update_layout(
    title=f'Probabilidad de victoria por equipo y semana · {SEASON}',
    template='plotly_white',
    height=900,
    xaxis_title='Semana',
    yaxis=dict(title='', autorange='reversed'),
)
fig.show()

## §7 · Trayectoria de ELO por equipo

Evolución del ELO de cada equipo a lo largo de la temporada, con los
resultados reales ya incorporados. Da clic en un equipo de la leyenda para
aislarlo (doble clic para volver a mostrar todos).

In [9]:
fig = go.Figure()

for t in sorted(elo_evolution_df['team'].unique()):
    t_df  = elo_evolution_df[elo_evolution_df['team'] == t].sort_values('week')
    color = AFC_COLOR if TEAM_CONF[t] == 'AFC' else NFC_COLOR
    fig.add_trace(go.Scatter(
        x=t_df['week'], y=t_df['elo_pre_week'],
        mode='lines+markers', name=t,
        line=dict(color=color, width=1.5),
        marker=dict(size=4),
        opacity=0.75,
        hovertemplate=f'<b>{t}</b><br>Semana %{{x}}<br>ELO: %{{y:.0f}}<extra></extra>',
    ))

fig.add_hline(y=GLOBAL_MEAN, line_dash='dot', line_color='gray',
              annotation_text=f'Media global ({GLOBAL_MEAN})')

fig.update_layout(
    title=f'Trayectoria de ELO por equipo · {SEASON} (AFC azul · NFC rojo)',
    xaxis_title='Semana', yaxis_title='ELO rating',
    template='plotly_white', height=650,
    legend=dict(font=dict(size=9)),
)
fig.show()

## §8 · Calendario interactivo por equipo

Selecciona un equipo en el menú desplegable para ver su probabilidad de
victoria semana a semana. Verde = ganó, rojo = perdió, gris = pendiente.

In [10]:
teams_sorted = sorted(final_elos.keys())
fig = go.Figure()

for i, t in enumerate(teams_sorted):
    v = team_schedule_view(games_df, t)
    marker_colors = [
        'gray' if not played else ('#2ecc71' if won == 1 else '#e74c3c')
        for played, won in zip(v['played'], v['team_won'])
    ]
    fig.add_trace(go.Scatter(
        x=v['week'], y=v['team_win_prob'],
        mode='lines+markers',
        name=t,
        visible=(i == 0),
        line=dict(color='#888', width=1),
        marker=dict(size=10, color=marker_colors, line=dict(width=1, color='white')),
        customdata=np.stack([v['opponent'], np.where(v['is_home'], 'Local', 'Visitante')], axis=-1),
        hovertemplate='Semana %{x}<br>Rival: %{customdata[0]} (%{customdata[1]})<br>Prob. victoria: %{y:.1%}<extra></extra>',
    ))

buttons = []
for i, t in enumerate(teams_sorted):
    visible = [j == i for j in range(len(teams_sorted))]
    buttons.append(dict(label=t, method='update',
                         args=[{'visible': visible}, {'title': f'Probabilidad de victoria por semana · {t} · {SEASON}'}]))

fig.update_layout(
    updatemenus=[dict(buttons=buttons, direction='down', x=1.15, y=1, showactive=True)],
    title=f'Probabilidad de victoria por semana · {teams_sorted[0]} · {SEASON}',
    xaxis_title='Semana', yaxis_title='Prob. de victoria',
    yaxis=dict(tickformat='.0%', range=[0, 1]),
    template='plotly_white', height=500,
)
fig.add_hline(y=0.5, line_dash='dash', line_color='lightgray')
fig.show()

In [15]:
wk = games_df[(games_df['week'] == 2) & games_df['played']].copy()
wk['pick_home'] = (wk['p_home_pre'] > 0.5).astype(int)

acc      = (wk['pick_home'] == wk['result']).mean()
brier    = ((wk['p_home_pre'] - wk['result'])**2).mean()
exp_hits = wk[['p_home_pre', 'p_away_pre']].max(axis=1).sum()   # aciertos esperados

print(f"Acc={acc:.1%} | BS={brier:.4f} | esperados={exp_hits:.1f} | reales={int(acc*len(wk))}")

Acc=62.5% | BS=0.2326 | esperados=10.3 | reales=10


In [20]:
played = games_df[games_df["played"] & games_df["week"].isin([1, 2])].copy()

played = played[played["home_score"] != played["away_score"]]   # empates fuera
played["home_won"]  = played["home_score"] > played["away_score"]
played["pick_home"] = played["p_home_pre"] > 0.5
played["correct"]   = played["pick_home"] == played["home_won"]

print(played.groupby("week")["correct"].agg(["sum", "count"]))
print("Accuracy total:", played["correct"].mean().round(3))

brier = ((played["p_home_pre"] - played["home_won"]) ** 2).mean()
print("Brier:", round(brier, 3), "(0.25 = lanzar una moneda)")

      sum  count
week            
1       9     16
2      10     16
Accuracy total: 0.594
Brier: 0.237 (0.25 = lanzar una moneda)


## Notas de uso

- **Cada semana**: vuelve a correr el notebook completo. `nflreadpy` trae los
  marcadores reales más recientes automáticamente; el ELO se actualiza solo
  con los partidos que ya se jugaron.
- Los CSVs en `data/processed/weekly/` quedan versionados por semana — útil
  para comparar cómo cambió la probabilidad de un juego conforme se acercaba
  (line sharpening) o para armar un post de LinkedIn semana a semana.
- Este motor usa el mismo ELO puro que el Notebook 04 (sin regresión
  logística), consistente con la decisión de diseño documentada en el
  Notebook 03 (`elo_diff` domina el poder predictivo).

In [36]:
"""
Gráficas verticales (1080x1920) para el reel "Pronosticando la NFL".
 
Uso en tu notebook:
    from reel_nfl_graficas import generar_todo
    generar_todo(games_df, semana=3, destacados=[("ARI", "SF"), ("BAL", "DAL"), ("MIN", "TB")])
 
Espera games_df con las columnas: season, week, home_team, away_team,
elo_home_pre, elo_away_pre, p_home_pre, played, home_score, away_score,
elo_home_post, elo_away_post.
"""
from pathlib import Path
 
import matplotlib.pyplot as plt
import pandas as pd
 
# ---------- Estilo (papel cuadriculado, como tu slide de LaLiga) ----------
FONDO = "#EFE6DC"
TINTA = "#1E1B18"
ACENTO = "#7A6A50"      # dorado/café: favorito, lo que sube
FRIO = "#C9CFE0"        # azul grisáceo: rival, lo secundario
ROJO = "#B5523B"        # lo que cae
GRIS = "#8C8578"
 
W, H, DPI = 9, 16, 120  # 1080 x 1920
# Zona segura: Instagram tapa arriba (~12%) y abajo (~18%)
SAFE_TOP, SAFE_BOTTOM = 0.84, 0.20
 
plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "text.color": TINTA,
    "axes.labelcolor": TINTA,
    "xtick.color": TINTA,
    "ytick.color": TINTA,
})
 
 
def _lienzo(etiqueta, titulo, pie="NFL · ELO"):
    fig = plt.figure(figsize=(W, H), dpi=DPI, facecolor=FONDO)
    # Cuadrícula de fondo
    bg = fig.add_axes([0, 0, 1, 1], zorder=-1)
    bg.set_facecolor(FONDO)
    bg.set_xlim(0, 9); bg.set_ylim(0, 16)
    for x in range(10):
        bg.axvline(x, color="#D9CEC0", lw=0.8)
    for y in range(17):
        bg.axhline(y, color="#D9CEC0", lw=0.8)
    bg.axis("off")
 
    fig.text(0.5, SAFE_TOP, etiqueta.upper(), ha="center", fontsize=20,
             color=ACENTO, fontweight="bold")
    fig.text(0.5, SAFE_TOP - 0.035, titulo, ha="center", va="top",
             fontsize=32, fontweight="bold", wrap=True)
    fig.text(0.06, SAFE_BOTTOM - 0.05, f"— {pie}", fontsize=14, color=GRIS)
    fig.text(0.94, SAFE_BOTTOM - 0.05, "@SebsBarroso", fontsize=14,
             color=GRIS, ha="right")
    return fig
 
 
def _guardar(fig, carpeta, nombre):
    Path(carpeta).mkdir(parents=True, exist_ok=True)
    ruta = Path(carpeta) / nombre
    fig.savefig(ruta, dpi=DPI, facecolor=FONDO)
    plt.close(fig)
    return ruta
 
 
# ---------- Datos derivados ----------
def elo_por_equipo(games_df):
    """Elo de pretemporada y Elo actual (solo partidos jugados)."""
    jug = games_df[games_df["played"]].sort_values("week")
    todos = games_df.sort_values("week")
 
    pre = pd.concat([
        todos[["week", "home_team", "elo_home_pre"]].set_axis(["week", "team", "elo"], axis=1),
        todos[["week", "away_team", "elo_away_pre"]].set_axis(["week", "team", "elo"], axis=1),
    ]).sort_values("week").groupby("team")["elo"].first()
 
    post = pd.concat([
        jug[["week", "home_team", "elo_home_post"]].set_axis(["week", "team", "elo"], axis=1),
        jug[["week", "away_team", "elo_away_post"]].set_axis(["week", "team", "elo"], axis=1),
    ]).sort_values("week").groupby("team")["elo"].last()
 
    df = pd.DataFrame({"pre": pre, "actual": post}).dropna()
    df["cambio"] = df["actual"] - df["pre"]
    return df
 
 
def picks_semana(games_df, semana):
    g = games_df[games_df["week"] == semana].copy()
    g["favorito"] = g.apply(lambda r: r["home_team"] if r["p_home_pre"] > 0.5 else r["away_team"], axis=1)
    g["rival"] = g.apply(lambda r: r["away_team"] if r["p_home_pre"] > 0.5 else r["home_team"], axis=1)
    g["p_fav"] = g["p_home_pre"].where(g["p_home_pre"] > 0.5, 1 - g["p_home_pre"])
    g["fav_local"] = g["p_home_pre"] > 0.5
    return g.sort_values("p_fav", ascending=False)
 
 
def record(games_df, semanas):
    j = games_df[games_df["played"] & games_df["week"].isin(semanas)].copy()
    j = j[j["home_score"] != j["away_score"]]
    j["home_won"] = j["home_score"] > j["away_score"]
    aciertos = ((j["p_home_pre"] > 0.5) == j["home_won"]).sum()
    solo_local = j["home_won"].sum()
    return int(aciertos), int(solo_local), len(j)
 
 
# ---------- Gráficas ----------
def g_record(games_df, semanas, carpeta):
    aciertos, local, n = record(games_df, semanas)
    fig = _lienzo("La prueba", f"Semanas {semanas[0]}–{semanas[-1]}:\n¿qué tal le fue al modelo?")
    fig.text(0.5, 0.56, f"{aciertos}/{n}", ha="center", fontsize=120,
             fontweight="bold", color=ACENTO)
    fig.text(0.5, 0.50, "PARTIDOS ACERTADOS", ha="center", fontsize=20, color=TINTA)
 
    ax = fig.add_axes([0.34, 0.29, 0.50, 0.14], facecolor="none")
    etiquetas = ["Elegir siempre\nal local", "El modelo"]
    valores = [local, aciertos]
    ax.barh(etiquetas, valores, color=[FRIO, ACENTO], height=0.6)
    for i, v in enumerate(valores):
        ax.text(v + 0.3, i, f"{v}/{n}", va="center", fontsize=22, fontweight="bold")
    ax.set_xlim(0, n)
    ax.tick_params(axis="y", labelsize=20, length=0)
    ax.set_xticks([])
    for s in ax.spines.values():
        s.set_visible(False)
    return _guardar(fig, carpeta, "01_record.png")
 
 
def g_movers(games_df, carpeta, n=5):
    df = elo_por_equipo(games_df).sort_values("cambio")
    sel = pd.concat([df.head(n), df.tail(n)])
    fig = _lienzo("Quién sube, quién cae", "Cambio de Elo\ndesde la pretemporada")
    ax = fig.add_axes([0.18, 0.26, 0.70, 0.44], facecolor="none")
    colores = [ACENTO if c > 0 else ROJO for c in sel["cambio"]]
    ax.barh(sel.index, sel["cambio"], color=colores, height=0.65)
    ax.axvline(0, color=TINTA, lw=1.5)
    lim = sel["cambio"].abs().max() * 1.35
    ax.set_xlim(-lim, lim)
    for i, (team, c) in enumerate(sel["cambio"].items()):
        ax.text(c + (lim * 0.03 if c > 0 else -lim * 0.03), i, f"{c:+.0f}",
                va="center", ha="left" if c > 0 else "right", fontsize=18, fontweight="bold")
    ax.tick_params(axis="y", labelsize=22, length=0)
    ax.set_xticks([])
    for s in ax.spines.values():
        s.set_visible(False)
    return _guardar(fig, carpeta, "02_suben_caen.png")
 
 
def g_top(games_df, carpeta, n=5):
    df = elo_por_equipo(games_df).sort_values("actual", ascending=False).head(n)[::-1]
    fig = _lienzo("Ranking del modelo", f"Top {n} según el Elo")
    ax = fig.add_axes([0.18, 0.30, 0.66, 0.38], facecolor="none")
    colores = [FRIO] * (n - 1) + [ACENTO]
    ax.barh(df.index, df["actual"] - 1400, left=1400, color=colores, height=0.6)
    for i, v in enumerate(df["actual"]):
        ax.text(v + 5, i, f"{v:.0f}", va="center", fontsize=22, fontweight="bold")
    ax.set_xlim(1400, df["actual"].max() + 60)
    ax.tick_params(axis="y", labelsize=26, length=0)
    ax.set_xticks([])
    for s in ax.spines.values():
        s.set_visible(False)
    fig.text(0.5, 0.26, "Promedio de la liga ≈ 1500", ha="center", fontsize=16, color=GRIS)
    return _guardar(fig, carpeta, "03_top.png")
 
 
def g_picks(games_df, semana, carpeta):
    g = picks_semana(games_df, semana)
    fig = _lienzo(f"Semana {semana}", "Mis pronósticos")
    top, alto = 0.72, 0.46
    fila = alto / len(g)
    for i, r in enumerate(g.itertuples()):
        y = top - i * fila
        sede = "en casa" if r.fav_local else "de visita"
        fig.text(0.10, y, r.favorito, fontsize=22, fontweight="bold", va="center")
        fig.text(0.27, y, f"vs {r.rival}", fontsize=18, color=GRIS, va="center")
        fig.text(0.47, y, sede, fontsize=13, color=GRIS, va="center")
        # barra de confianza
        ax = fig.add_axes([0.62, y - fila * 0.28, 0.22, fila * 0.56], facecolor="none")
        ax.barh([0], [r.p_fav], color=ACENTO if r.p_fav >= 0.6 else FRIO, height=1)
        ax.barh([0], [1 - r.p_fav], left=r.p_fav, color="#E2D8CB", height=1)
        ax.set_xlim(0, 1); ax.axis("off")
        fig.text(0.86, y, f"{r.p_fav:.0%}", fontsize=18, fontweight="bold", va="center")
    return _guardar(fig, carpeta, f"04_picks_semana{semana}.png")
 
 
def g_duelo(games_df, semana, visita, local, carpeta, etiqueta="Partido destacado"):
    g = games_df[(games_df["week"] == semana) & (games_df["home_team"] == local)
                 & (games_df["away_team"] == visita)]
    if g.empty:
        raise ValueError(f"No encontré {visita} @ {local} en la semana {semana}")
    p_local = float(g["p_home_pre"].iloc[0])
    p_visita = 1 - p_local
    fig = _lienzo(etiqueta, f"{visita} @ {local}")
 
    ax = fig.add_axes([0.08, 0.46, 0.84, 0.07], facecolor="none")
    c_vis = ACENTO if p_visita > p_local else FRIO
    c_loc = ACENTO if p_local > p_visita else FRIO
    ax.barh([0], [p_visita], color=c_vis, height=1)
    ax.barh([0], [p_local], left=p_visita, color=c_loc, height=1)
    ax.axvline(0.5, color=TINTA, lw=1.5, ls="--")
    ax.set_xlim(0, 1); ax.axis("off")
 
    fig.text(0.08, 0.58, visita, fontsize=48, fontweight="bold")
    fig.text(0.92, 0.58, local, fontsize=48, fontweight="bold", ha="right")
    fig.text(0.08, 0.41, f"{p_visita:.0%}", fontsize=40, fontweight="bold", va="top",
             color=ACENTO if p_visita > p_local else GRIS)
    fig.text(0.92, 0.41, f"{p_local:.0%}", fontsize=40, fontweight="bold", va="top",
             ha="right", color=ACENTO if p_local > p_visita else GRIS)
    fig.text(0.08, 0.34, "visitante", fontsize=16, color=GRIS)
    fig.text(0.92, 0.34, "local", fontsize=16, color=GRIS, ha="right")
    return _guardar(fig, carpeta, f"05_duelo_{visita}_{local}.png")
 
 
def trayectoria_elo(games_df):
    """Tabla equipo x punto (Pre, S1, S2, ...) solo con semanas jugadas."""
    pre = elo_por_equipo(games_df)["pre"]
    jug = games_df[games_df["played"]]
    largo = pd.concat([
        jug[["week", "home_team", "elo_home_post"]].set_axis(["week", "team", "elo"], axis=1),
        jug[["week", "away_team", "elo_away_post"]].set_axis(["week", "team", "elo"], axis=1),
    ])
    tabla = largo.pivot_table(index="team", columns="week", values="elo", aggfunc="last")
    tabla = tabla.sort_index(axis=1).ffill(axis=1)       # semanas de descanso
    tabla.insert(0, 0, pre)
    tabla[0] = tabla[0].fillna(tabla.iloc[:, 1])
    tabla = tabla.T.ffill().T
    tabla.columns = ["Pre"] + [f"S{int(c)}" for c in tabla.columns[1:]]
    return tabla
 
 
def g_evolucion(games_df, carpeta, n=3):
    t = trayectoria_elo(games_df)
    cambio = t.iloc[:, -1] - t.iloc[:, 0]
    suben = cambio.sort_values().tail(n).index
    caen = cambio.sort_values().head(n).index
    ultima = t.columns[-1]
 
    fig = _lienzo("Evolución del Elo", f"Así van los 32\nhasta la {ultima.replace('S', 'semana ')}")
    ax = fig.add_axes([0.14, 0.27, 0.66, 0.46], facecolor="none")
    x = range(len(t.columns))
 
    for team, fila in t.iterrows():                       # el resto, en gris
        if team not in suben and team not in caen:
            ax.plot(x, fila.values, color="#BDB3A5", lw=1.5, alpha=0.7, zorder=1)
 
    etiquetas = []
    for team in list(suben) + list(caen):
        color = ACENTO if team in suben else ROJO
        ax.plot(x, t.loc[team].values, color=color, lw=4, marker="o", ms=8, zorder=3)
        etiquetas.append([t.loc[team].iloc[-1], team, color])
 
    # Etiquetas al final de cada línea, separadas para que no se encimen
    etiquetas.sort()
    sep = (t.values.max() - t.values.min()) * 0.045
    for i in range(1, len(etiquetas)):
        etiquetas[i][0] = max(etiquetas[i][0], etiquetas[i - 1][0] + sep)
    for y, team, color in etiquetas:
        ax.text(len(t.columns) - 1 + 0.12, y, team, color=color, fontsize=20,
                fontweight="bold", va="center")
 
    ax.axhline(1500, color=TINTA, lw=1, ls="--", alpha=0.6)
    ax.text(0, 1500, " promedio 1500", fontsize=13, color=GRIS, va="bottom")
    ax.set_xticks(list(x))
    ax.set_xticklabels(t.columns, fontsize=20)
    ax.tick_params(axis="y", labelsize=16, length=0)
    ax.set_xlim(-0.2, len(t.columns) - 1 + 0.2)
    for s in ["top", "right", "bottom"]:
        ax.spines[s].set_visible(False)
    ax.spines["left"].set_color(GRIS)
    fig.text(0.49, 0.235, f"▲ los {n} que más suben", ha="right", fontsize=15, color=ACENTO)
    fig.text(0.53, 0.235, f"▼ los {n} que más caen", ha="left", fontsize=15, color=ROJO)
    return _guardar(fig, carpeta, "02b_evolucion_elo.png")
 
 
def g_barras_local(games_df, semana, carpeta):
    g = games_df[games_df["week"] == semana].copy()
    g = g.sort_values("p_home_pre")                       # mayor arriba
    g["etiqueta"] = g["away_team"] + " @ " + g["home_team"]
 
    fig = _lienzo(f"Semana {semana}", "¿Gana el local?")
    ax = fig.add_axes([0.26, 0.26, 0.60, 0.48], facecolor="none")
    colores = [ACENTO if p > 0.5 else FRIO for p in g["p_home_pre"]]
    ax.barh(g["etiqueta"], g["p_home_pre"], color=colores, height=0.7)
    ax.axvline(0.5, color=TINTA, lw=1.5, ls="--")
    for i, p in enumerate(g["p_home_pre"]):
        ax.text(p + 0.015, i, f"{p:.0%}", va="center", fontsize=16, fontweight="bold")
    ax.set_xlim(0, 1)
    ax.set_xticks([0, 0.5, 1])
    ax.set_xticklabels(["0%", "50%", "100%"], fontsize=15)
    ax.tick_params(axis="y", labelsize=17, length=0)
    for s in ["top", "right", "left"]:
        ax.spines[s].set_visible(False)
    ax.spines["bottom"].set_color(GRIS)
    fig.text(0.5, 0.215, "Probabilidad de que gane el equipo local",
             ha="center", fontsize=15, color=GRIS)
    fig.text(0.49, 0.195, "■ favorito el local", ha="right", fontsize=15, color=ACENTO)
    fig.text(0.53, 0.195, "■ favorito el visitante", ha="left", fontsize=15, color="#8E97B5")
    return _guardar(fig, carpeta, f"04b_prob_local_semana{semana}.png")
 
 
def generar_todo(games_df, semana, destacados=(), carpeta="reel_semana"):
    carpeta = f"{carpeta}{semana}"
    previas = list(range(1, semana))
    rutas = [
        g_record(games_df, previas, carpeta),
        g_movers(games_df, carpeta),
        g_evolucion(games_df, carpeta),
        g_top(games_df, carpeta),
        g_picks(games_df, semana, carpeta),
        g_barras_local(games_df, semana, carpeta),
    ]
    nombres = ["El más seguro", "La moneda al aire", "La sorpresa"]
    for i, (vis, loc) in enumerate(destacados):
        etiqueta = nombres[i] if i < len(nombres) else "Partido destacado"
        rutas.append(g_duelo(games_df, semana, vis, loc, carpeta, etiqueta))
    for r in rutas:
        print("✓", r)
    return rutas

In [37]:
generar_todo(games_df, semana=3, destacados=[("ARI", "SF"), ("BAL", "DAL"), ("MIN", "TB")])

✓ reel_semana3/01_record.png
✓ reel_semana3/02_suben_caen.png
✓ reel_semana3/02b_evolucion_elo.png
✓ reel_semana3/03_top.png
✓ reel_semana3/04_picks_semana3.png
✓ reel_semana3/04b_prob_local_semana3.png
✓ reel_semana3/05_duelo_ARI_SF.png
✓ reel_semana3/05_duelo_BAL_DAL.png
✓ reel_semana3/05_duelo_MIN_TB.png


[PosixPath('reel_semana3/01_record.png'),
 PosixPath('reel_semana3/02_suben_caen.png'),
 PosixPath('reel_semana3/02b_evolucion_elo.png'),
 PosixPath('reel_semana3/03_top.png'),
 PosixPath('reel_semana3/04_picks_semana3.png'),
 PosixPath('reel_semana3/04b_prob_local_semana3.png'),
 PosixPath('reel_semana3/05_duelo_ARI_SF.png'),
 PosixPath('reel_semana3/05_duelo_BAL_DAL.png'),
 PosixPath('reel_semana3/05_duelo_MIN_TB.png')]